# Location Selection

In [13]:
from desdeo.problem import Constant, Variable, Problem, Objective, VariableTypeEnum, Constraint, TensorConstant, TensorVariable, ConstraintTypeEnum
import numpy as np

# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Model inputs


In [14]:

# Mininum expected attendance to be worth visiting 
min_att = 15

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.00

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 6.0
# How long the event is (hours)
hours_per_event = 4
driver_salary_per_hour = 19
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour

# Food desert threshold (miles) (TODO this should be in minutes)
raw_food_desert_threshold = 15




## Load and process Constants

In [15]:
from slugify import slugify
import pandas as pd

def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)

home = "Ada"
# Read the adjacency matrix (distance in miles)
adjDist = pd.read_csv("adjacencyMatrixDist.csv", index_col=0)
dist2home = adjDist.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"
display(dist2home)

# Read the adjacency matrix (travel time in minutes)
adjTTime = pd.read_csv("adjacencyMatrixTravelTime.csv", index_col=0)
display(adjTTime)

# Read the cities 
cities = pd.read_csv("cities.csv")
events = pd.read_csv("events.csv")



,dist2home
city,
Ada,0.0
Lima,16.2
Kenton,15.5
Delphos,31.8
Bluffton,12.0
Spencerville,30.8
Elida,23.5
Forest,18.4
Alger,5.4


,Ada,Alger,Bluffton,Cairo,Columbus Grove,Continental,Cridersville,Delphos,Dunkirk,Elida,...,Mount Victory,New Bremen,New Knoxville,Ottawa,Ottoville,Pandora,Saint Marys,Spencerville,Wapakoneta,Waynesfield
Ada,0,9,16,20,25,48,28,34,16,30,...,34,52,46,34,37,26,45,43,36,25
Alger,9,0,22,26,31,54,29,40,20,35,...,33,53,46,40,43,32,46,45,36,19
Bluffton,16,22,0,17,15,39,25,31,23,27,...,48,49,42,23,34,13,42,40,32,32
Cairo,21,26,17,0,15,39,20,18,27,13,...,48,44,38,17,34,13,37,26,28,28
Columbus Grove,26,31,15,9,0,26,26,24,32,20,...,57,50,43,9,22,8,43,34,33,33
Continental,48,53,40,30,25,0,49,30,54,33,...,80,68,64,19,21,32,58,42,56,56
Cridersville,28,29,25,21,26,49,0,29,39,17,...,49,30,23,35,34,30,23,18,13,17
Delphos,34,39,31,18,24,30,29,0,40,12,...,66,40,35,32,11,33,30,14,33,42
Dunkirk,16,20,23,27,32,54,39,41,0,36,...,28,63,56,41,44,33,56,50,46,36
Elida,30,35,26,13,21,33,17,12,36,0,...,61,41,33,28,17,29,32,16,24,33


In [16]:
# Create event table
events = pd.merge(cities, events, on="city")
events.loc[:,"expectedAttendance"] = (events.loc[:,"pop"] * events.loc[:,"attendanceRate"]).astype(int)

events.loc[:, "event_id"] = events.apply(lambda row: slugify(f'{row["city"]} {row["site"]} {no_nan(row["event"])}'), axis=1)

# Add the distance to home for each event 
events = pd.merge(events, dist2home, on="city")

display(events)

,city,lat,long,pop,site,event,attendanceRate,expectedAttendance,event_id,dist2home
0,Ada,40.768056,-83.825278,5334,Public Library,NaN,0.002,10,ada-public-library,0.0
1,Lima,40.746389,-84.123333,35579,Mercy Health Thrift,NaN,0.002,71,lima-mercy-health-thrift,16.2
2,Lima,40.746389,-84.123333,35579,Habitat For Humanity,NaN,0.002,71,lima-habitat-for-humanity,16.2
3,Lima,40.746389,-84.123333,35579,Our Daily Bread,NaN,0.002,71,lima-our-daily-bread,16.2
4,Lima,40.746389,-84.123333,35579,St. Mark’s Methodist,Community Meal,0.002,71,lima-st-marks-methodist-community-meal,16.2
5,Lima,40.746389,-84.123333,35579,Christian Corner Community Center,NaN,0.002,71,lima-christian-corner-community-center,16.2
6,Kenton,40.646667,-83.622500,7947,Seton Hall,NaN,0.002,15,kenton-seton-hall,15.5
7,Kenton,40.646667,-83.622500,7947,Hardincrest,NaN,0.002,15,kenton-hardincrest,15.5
8,Kenton,40.646667,-83.622500,7947,YMCA,NaN,0.002,15,kenton-ymca,15.5
9,Delphos,40.861111,-84.350000,7117,Public Library,NaN,0.002,14,delphos-public-library,31.8


### Close cities

In [17]:

close_cities = adjTTime < raw_food_desert_threshold
event2city = events.loc[:,["city", "event_id"]].merge(close_cities, left_on="city", right_index=True)
event2city = (event2city.iloc[:,2:].values).astype(int)
A_raw = event2city.T
A_raw_list = A_raw.tolist()
A_raw_list

[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
 [0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Constants

In [30]:
event_count = events.shape[0]

# Expected attenance 
raw_attendance = events.loc[:,["expectedAttendance"]] 
raw_over_attendance = (raw_attendance < min_att).astype(int)

ea = TensorConstant(name="Expected attendance",
                                 symbol="ea", 
                                 type="int",
                                 shape=raw_attendance.T.shape,
                                 values=raw_attendance.T.values.tolist())
display(ea)
print(raw_over_attendance.T.shape)

eoa = TensorConstant(name="Expected over attendance", 
                    symbol="eoa", 
                    type="integer",
                    shape=raw_over_attendance.T.shape,
                    values=raw_over_attendance.T.values.tolist())

display(eoa)
print(raw_over_attendance.T.shape)


TensorConstant(name='Expected attendance', symbol='ea', shape=[1, 18], values=['List', ['List', 10, 71, 71, 71, 71, 71, 15, 15, 15, 14, 7, 3, 2, 1, 1, 1, 1, 1]])

(1, 18)


TensorConstant(name='Expected over attendance', symbol='eoa', shape=[1, 18], values=['List', ['List', 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

(1, 18)


## Variables

In [29]:
ev = TensorVariable(
  name="Events visited",
  symbol="ev",
  variable_type=VariableTypeEnum.integer,
  shape=[events.shape[0],1],
  lowerbounds=0,
  upperbounds=1,
  initial_value=0)

ev

TensorVariable(name='Events visited', symbol='ev', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[18, 1], lowerbounds=0, upperbounds=1, initial_values=None)

## Objective

In [39]:

total_patients = Objective(
    name = "Maximize total patients visited",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = "Sum(ea@ev)"
)

# Overstaffed events
over_staffed_events = Objective(
    name = "Minimize the number of overstaffed events",
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(eoa@ev)"
)

## Problem

In [40]:
prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        type="linear",
        constants=[eoa, ea],
        variables=[ev], #+ [aux_desert_var],
        objectives=[total_patients, over_staffed_events]
    )

## Ideal/nadir

In [41]:
# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(events.loc[:,"expectedAttendance"]))

# f_2 ideal is having no over staffed events
# f_2 naird is having visiting all locations with over staffed events
all_ose = int(np.sum(raw_over_attendance))



prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose
        }
    )

print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")


Ideal values: {'f_1': 441, 'f_2': 0}
Nadir values: {'f_1': 0, 'f_2': 10}


## RPM solver


In [ ]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions

reference_point = {"f_1": 200, "f_2": 2}
display(prob)
results = rpm_solve_solutions(prob, reference_point=reference_point)


Problem(name='Simple site selection', description='Simple implementation of the site selection problem', constants=[TensorConstant(name='Expected over attendance', symbol='eoa', shape=[1, 18], values=['List', ['List', 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), TensorConstant(name='Expected attendance', symbol='ea', shape=[1, 18], values=['List', ['List', 10, 71, 71, 71, 71, 71, 15, 15, 15, 14, 7, 3, 2, 1, 1, 1, 1, 1]])], variables=[TensorVariable(name='Events visited', symbol='ev', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[18, 1], lowerbounds=0, upperbounds=1, initial_values=None)], objectives=[Objective(name='Maximize total patients visited', symbol='f_1', unit=None, func=['Sum', ['MatMul', 'ea', 'ev']], simulator_path=None, surrogates=None, maximize=True, ideal=441, nadir=0, objective_type=<ObjectiveTypeEnum.analytical: 'analytical'>, is_linear=False, is_convex=False, is_twice_differentiable=True, scenario_keys=None), Objective(name='Minimize the numbe

In [43]:

for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print("---")

results

Solution 1:
Objective function values 		 {'f_1': 400.0, 'f_2': 0.0}
Decision variable values 		 {'ev': [[0.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], '_alpha': -0.19999998999000201}
Constraint values 			 {'f_1_con': -0.25351474821064485, 'f_2_con': 9.989999993909038e-09}
---
Solution 2:
Objective function values 		 {'f_1': 414.0, 'f_2': 1.0}
Decision variable values 		 {'ev': [[0.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], '_alpha': -0.031723366494000184}
Constraint values 			 {'f_1_con': 9.989999917581205e-09, 'f_2_con': -0.06827662350600083}
---
Solution 3:
Objective function values 		 {'f_1': 441.0, 'f_2': 10.0}
Decision variable values 		 {'ev': [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], '_alpha': -0.5464852695217793}
Constraint values 			 

[SolverResults(optimal_variables={'ev': [[0.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], '_alpha': -0.19999998999000201}, optimal_objectives={'f_1': 400.0, 'f_2': 0.0}, constraint_values={'f_1_con': -0.25351474821064485, 'f_2_con': 9.989999993909038e-09}, extra_func_values=None, scalarization_values={'_asf': -0.2000008970194784}, success=True, message="Pyomo solver status is: 'ok', with termination condition: 'optimal'."),
 SolverResults(optimal_variables={'ev': [[0.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0.0]], '_alpha': -0.031723366494000184}, optimal_objectives={'f_1': 414.0, 'f_2': 1.0}, constraint_values={'f_1_con': 9.989999917581205e-09, 'f_2_con': -0.06827662350600083}, extra_func_values=None, scalarization_values={'_asf': -0.03172420526951826}, success=True, message="Pyomo solver status is: 'ok', with termination condition: 